# 01 — Pipeline de Datos: Extracción Programática, Carga Relacional y Análisis SQL

Este cuaderno ejecuta y documenta de principio a fin la ingeniería de datos del proyecto **Coyuntura Macroeconómica y Proyección del IMACEC**:

1. **Extracción e Ingesta (`src/extraccion.py` $\rightarrow$ `src/carga_db.py`):** Descarga programática de **17 series económicas** desde la API oficial **SieteRestWS del Banco Central de Chile** (con soporte de modo `--demo` sin credenciales) y persistencia idempotente en SQLite (`data/coyuntura.db`).
2. **Auditoría del Esquema Relacional (`sql/01_esquema.sql`):** Verificación de las tres tablas normalizadas (`temas`, `series` y `observaciones`) y sus índices de optimización.
3. **Consolidación Analítica en SQL (`sql/02_consulta.sql`):** Homologación de frecuencias diarias a mensuales (`GROUP BY`), cálculo de variaciones interanuales (`LAG`) y promedios móviles de 12 meses (`AVG OVER`) mediante funciones de ventana y cruces `JOIN`.

## 1. Configuración del Entorno, Extracción API y Carga en SQLite

**Síntesis:** En esta etapa configuramos la ruta hacia la carpeta `src/` del proyecto, definimos la ventana temporal oficial del estudio (`2010-01-01` a `2026-08-31`) y ejecutamos `carga_db.main()`. Esta función crea el esquema relacional de forma idempotente, registra el catálogo de 17 indicadores, descarga las observaciones desde la API del Banco Central de Chile y emite un reporte de auditoría en consola. *(Si aún no cuentas con credenciales en `.env`, cambia `DEMO = True` para probar el flujo con datos sintéticos).*

In [6]:
import sys
from pathlib import Path
import pandas as pd
from sqlalchemy import text

# 1. Detectar automáticamente la raíz del repositorio y agregar `src/` al PATH
raiz_repo = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
ruta_src = raiz_repo / "src"
if str(ruta_src) not in sys.path:
    sys.path.insert(0, str(ruta_src))

import config
import carga_db

# 2. Parámetros de ejecución del pipeline
# Cambiar DEMO = True únicamente si se desea probar sin credenciales API en .env
DEMO = False
FECHA_INICIO = config.FECHA_INICIO   # "2010-01-01"
FECHA_CORTE = "2026-08-31"           # Incluye todos los días hábiles de agosto 2026

# 3. Ejecutar el flujo completo: esquema SQL -> catálogo -> extracción API -> UPSERT en SQLite
carga_db.main(demo=DEMO, desde=FECHA_INICIO, hasta=FECHA_CORTE)

 PIPELINE DE INGESTA MACROECONÓMICA -> SQLITE
  • Fuente  : API Banco Central de Chile (SieteRestWS)
  • Ventana : 2010-01-01 hasta 2026-08-31

[OK] Esquema relacional SQL verificado / creado.
[OK] Catálogo sincronizado: 17 series registradas.
  [OK] Imacec empalmado                 |   200 obs. | hasta 2026-08-01
  [OK] Imacec minero                    |   200 obs. | hasta 2026-08-01
  [OK] Imacec industria                 |   200 obs. | hasta 2026-08-01
  [OK] Imacec resto bienes              |   200 obs. | hasta 2026-08-01
  [OK] Imacec comercio                  |   200 obs. | hasta 2026-08-01
  [OK] Imacec servicios                 |   200 obs. | hasta 2026-08-01
  [OK] Imacec impuestos                 |   164 obs. | hasta 2026-08-01
  [OK] Precio cobre                     |   200 obs. | hasta 2026-08-01
  [OK] Tipo cambio observado            |  4150 obs. | hasta 2026-08-31
  [OK] Exportaciones cobre              |   200 obs. | hasta 2026-08-01
  [OK] Exportaciones litio          

## 2. Verificación de la Estructura Relacional en SQLite

**Síntesis:** Una vez poblada la base de datos `data/coyuntura.db`, abrimos una conexión mediante SQLAlchemy y consultamos el catálogo interno de SQLite (`sqlite_master`) para confirmar que las **3 tablas relacionales** (`temas`, `series`, `observaciones`) y los **índices de rendimiento** (`idx_obs_fecha`, `idx_series_tema`) fueron creados correctamente.

In [7]:
# 1. Inicializar el motor de conexión SQLAlchemy con integridad referencial activa
engine = carga_db.obtener_engine()

# 2. Consultar los objetos estructurales (tablas e índices) creados en coyuntura.db
consulta_objetos = text("""
    SELECT
        type AS tipo_objeto,
        name AS nombre_objeto,
        tbl_name AS tabla_asociada
    FROM sqlite_master
    WHERE type IN ('table', 'index')
      AND name NOT LIKE 'sqlite_%'
    ORDER BY type DESC, tbl_name, name;
""")

with engine.connect() as conn:
    objetos_db = pd.read_sql(consulta_objetos, conn)

print(f"Base de datos verificada en: {config.RUTA_DB}")
objetos_db

Base de datos verificada en: C:\Users\mpalm\Downloads\portafolio grupo 12\Portafolio_grupo_12_Imacec\data\coyuntura.db


,tipo_objeto,nombre_objeto,tabla_asociada
0,table,observaciones,observaciones
1,table,series,series
2,table,temas,temas
3,index,idx_obs_fecha,observaciones
4,index,idx_series_tema,series


## 3. Inspección del Catálogo Macroeconómico (`JOIN` entre `series` y `temas`)

**Síntesis:** Ejecutamos una consulta relacional `JOIN` que cruza la tabla de catálogo `series` con la tabla clasificatoria `temas`. Esto permite auditar las **17 series económicas** registradas, verificando su código oficial en el Banco Central de Chile, fuente institucional, frecuencia original (`M` mensual o `D` diaria), unidad de medida y dimensión analítica asignada.

In [8]:
# Consulta relacional JOIN para inspeccionar el catálogo completo de 17 indicadores
consulta_catalogo = text("""
    SELECT
        t.id_tema,
        t.tema,
        s.id_serie,
        s.nombre AS serie,
        s.fuente,
        s.frecuencia,
        s.unidad
    FROM series AS s
    JOIN temas AS t ON t.id_tema = s.id_tema
    ORDER BY t.id_tema, s.nombre;
""")

with engine.connect() as conn:
    df_catalogo = pd.read_sql(consulta_catalogo, conn)

print(f"Total de series catalogadas: {len(df_catalogo)} en {df_catalogo['tema'].nunique()} áreas temáticas.")
df_catalogo

Total de series catalogadas: 17 en 5 áreas temáticas.


,id_tema,tema,id_serie,serie,fuente,frecuencia,unidad
0,1,Actividad,F032.IMC.IND.Z.Z.EP18.COM.Z.0.M,Imacec comercio,Banco Central de Chile,M,Indice 2018=100
1,1,Actividad,F032.IMC.IND.Z.Z.EP18.Z.Z.0.M,Imacec empalmado,Banco Central de Chile,M,Indice 2018=100
2,1,Actividad,F032.IMC.IND.Z.Z.EP18.IMP.Z.0.M,Imacec impuestos,Banco Central de Chile,M,Indice 2018=100
3,1,Actividad,F032.IMC.IND.Z.Z.EP18.04.Z.0.M,Imacec industria,Banco Central de Chile,M,Indice 2018=100
4,1,Actividad,F032.IMC.IND.Z.Z.EP18.03.Z.0.M,Imacec minero,Banco Central de Chile,M,Indice 2018=100
5,1,Actividad,F032.IMC.IND.Z.Z.EP18.RB.Z.0.M,Imacec resto bienes,Banco Central de Chile,M,Indice 2018=100
6,1,Actividad,F032.IMC.IND.Z.Z.EP18.SERV.Z.0.M,Imacec servicios,Banco Central de Chile,M,Indice 2018=100
7,2,Sector Externo,F068.B1.FLU.B.0.C.N.Z.Z.Z.Z.6.0.M,Exportaciones agropecuarias,Banco Central de Chile,M,Millones USD
8,2,Sector Externo,F068.B1.FLU.A1.0.C.N.Z.Z.Z.Z.6.0.M,Exportaciones cobre,Banco Central de Chile,M,Millones USD
9,2,Sector Externo,F068.B1.FLU.A8.0.C.N.Z.Z.Z.Z.6.0.M,Exportaciones litio,Banco Central de Chile,M,Millones USD


## 4. Auditoría de Cobertura Temporal y Completitud (`GROUP BY`)

**Síntesis:** Mediante una agregación con `GROUP BY` sobre la tabla de hechos `observaciones` cruzada con `series` y `temas`, auditamos la fecha inicial (`MIN`), la fecha de cierre (`MAX`) y el volumen total de registros almacenados (`COUNT`) para cada indicador. Esto confirma que las series mensuales cuentan con 200 observaciones (`2010-01` a `2026-08`) y las series diarias (TPM y Tipo de Cambio Observado) superan los 4.100 registros hábiles.

In [9]:
# Agregación GROUP BY para verificar cobertura histórica y recuento de registros por serie
consulta_cobertura = text("""
    SELECT
        t.tema,
        s.nombre AS serie,
        s.frecuencia,
        MIN(o.fecha) AS primera_obs,
        MAX(o.fecha) AS ultima_obs,
        COUNT(o.valor) AS total_observaciones
    FROM observaciones AS o
    JOIN series AS s ON s.id_serie = o.id_serie
    JOIN temas AS t ON t.id_tema = s.id_tema
    WHERE o.fecha BETWEEN '2010-01-01' AND '2026-08-31'
    GROUP BY t.id_tema, t.tema, s.nombre, s.frecuencia
    ORDER BY t.id_tema, s.nombre;
""")

with engine.connect() as conn:
    df_cobertura = pd.read_sql(consulta_cobertura, conn)

print(f"Total acumulado de observaciones válidas en el período: {df_cobertura['total_observaciones'].sum():,}")
df_cobertura

Total acumulado de observaciones válidas en el período: 11,263


,tema,serie,frecuencia,primera_obs,ultima_obs,total_observaciones
0,Actividad,Imacec comercio,M,2010-01-01,2026-08-01,200
1,Actividad,Imacec empalmado,M,2010-01-01,2026-08-01,200
2,Actividad,Imacec impuestos,M,2013-01-01,2026-08-01,164
3,Actividad,Imacec industria,M,2010-01-01,2026-08-01,200
4,Actividad,Imacec minero,M,2010-01-01,2026-08-01,200
5,Actividad,Imacec resto bienes,M,2010-01-01,2026-08-01,200
6,Actividad,Imacec servicios,M,2010-01-01,2026-08-01,200
7,Sector Externo,Exportaciones agropecuarias,M,2010-01-01,2026-08-01,200
8,Sector Externo,Exportaciones cobre,M,2010-01-01,2026-08-01,200
9,Sector Externo,Exportaciones litio,M,2010-01-01,2026-08-01,200


## 5. Transformación Analítica con SQL: Agregación Mensual y Funciones de Ventana

**Síntesis:** Replicamos la consulta principal definida en `sql/02_consulta.sql` y utilizada por `src/dashboard.py`. A través de expresiones de tabla común (`WITH`), el motor SQL ejecuta tres operaciones clave directamente sobre la base relacional:
1. **`GROUP BY`:** Colapsa las series diarias (TPM y Tipo de Cambio Observado) a promedios mensuales homogéneos (`AVG`).
2. **Funciones de ventana (`LAG` y `AVG OVER`):** Calcula la **variación porcentual interanual a 12 meses** (`var_12m_pct`) y el **promedio móvil de 12 meses** (`pm_12m`) particionando por serie y ordenando cronológicamente.
3. **`JOIN`:** Incorpora los metadatos descriptivos desde `series` y `temas`, dejando el panel listo para el análisis coyuntural y la modelación econométrica en el cuaderno `02_coyuntura_y_modelos.ipynb`.

In [10]:
# Consulta analítica completa (GROUP BY + LAG 12M + AVG OVER 12M + JOIN)
consulta_analitica = text("""
    WITH mensual AS (
        -- Paso 1: Homologar todas las series a frecuencia mensual (promedios para series diarias)
        SELECT
            o.id_serie,
            strftime('%Y-%m', o.fecha) AS mes,
            AVG(o.valor) AS valor_mensual
        FROM observaciones AS o
        WHERE o.fecha BETWEEN '2010-01-01' AND '2026-08-31'
        GROUP BY o.id_serie, mes
    ),
    indicadores AS (
        -- Paso 2: Calcular variación interanual (LAG 12) y tendencia móvil (AVG OVER 12M)
        SELECT
            m.id_serie,
            m.mes,
            m.valor_mensual,
            100.0 * (
                m.valor_mensual / NULLIF(
                    LAG(m.valor_mensual, 12) OVER (
                        PARTITION BY m.id_serie
                        ORDER BY m.mes
                    ),
                    0
                ) - 1.0
            ) AS var_12m_pct,
            AVG(m.valor_mensual) OVER (
                PARTITION BY m.id_serie
                ORDER BY m.mes
                ROWS BETWEEN 11 PRECEDING AND CURRENT ROW
            ) AS pm_12m
        FROM mensual AS m
    )
    -- Paso 3: Enriquecer con nombres y áreas temáticas mediante JOIN
    SELECT
        t.tema,
        i.id_serie,
        s.nombre AS serie,
        s.unidad,
        i.mes || '-01' AS fecha,
        ROUND(i.valor_mensual, 3) AS valor_mensual,
        ROUND(i.var_12m_pct, 2) AS var_12m_pct,
        ROUND(i.pm_12m, 3) AS pm_12m
    FROM indicadores AS i
    JOIN series AS s ON s.id_serie = i.id_serie
    JOIN temas AS t ON t.id_tema = s.id_tema
    ORDER BY t.id_tema, s.nombre, i.mes;
""")

with engine.connect() as conn:
    df_panel_sql = pd.read_sql(consulta_analitica, conn, parse_dates=["fecha"])

# Inspeccionar los registros más recientes del IMACEC agregado como verificación final
print(f"Panel consolidado generado: {len(df_panel_sql):,} filas x {df_panel_sql.shape[1]} columnas.")
df_panel_sql[df_panel_sql["id_serie"] == config.ID_OBJETIVO].tail(12)

Panel consolidado generado: 3,362 filas x 8 columnas.


,tema,id_serie,serie,unidad,fecha,valor_mensual,var_12m_pct,pm_12m
388,Actividad,F032.IMC.IND.Z.Z.EP18.Z.Z.0.M,Imacec empalmado,Indice 2018=100,2025-09-01,109.833,3.30,113.375
389,Actividad,F032.IMC.IND.Z.Z.EP18.Z.Z.0.M,Imacec empalmado,Indice 2018=100,2025-10-01,114.988,2.06,113.568
390,Actividad,F032.IMC.IND.Z.Z.EP18.Z.Z.0.M,Imacec empalmado,Indice 2018=100,2025-11-01,118.447,1.37,113.702
391,Actividad,F032.IMC.IND.Z.Z.EP18.Z.Z.0.M,Imacec empalmado,Indice 2018=100,2025-12-01,129.202,1.27,113.836
392,Actividad,F032.IMC.IND.Z.Z.EP18.Z.Z.0.M,Imacec empalmado,Indice 2018=100,2026-01-01,111.131,-0.94,113.748
393,Actividad,F032.IMC.IND.Z.Z.EP18.Z.Z.0.M,Imacec empalmado,Indice 2018=100,2026-02-01,104.260,-0.35,113.717
394,Actividad,F032.IMC.IND.Z.Z.EP18.Z.Z.0.M,Imacec empalmado,Indice 2018=100,2026-03-01,120.420,0.13,113.730
395,Actividad,F032.IMC.IND.Z.Z.EP18.Z.Z.0.M,Imacec empalmado,Indice 2018=100,2026-04-01,114.708,-0.77,113.656
396,Actividad,F032.IMC.IND.Z.Z.EP18.Z.Z.0.M,Imacec empalmado,Indice 2018=100,2026-05-01,112.157,-1.34,113.529
397,Actividad,F032.IMC.IND.Z.Z.EP18.Z.Z.0.M,Imacec empalmado,Indice 2018=100,2026-06-01,110.922,2.31,113.738
